In [15]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC))

In [16]:
#from app.pipeline import run_pipeline
#run_pipeline()

In [17]:
from __future__ import annotations

#環境非依存処理
from ifc_ingestor.ingestor import load_ifc_bundle
from bdns_extractor.extractor import extract_bdns_tags


from typing import Dict, Any
from app.contracts import BrickGraph
from ifc_ingestor.ingestor import load_ifc_bundle
from bdns_extractor.extractor import extract_bdns_tags
from class_mapper.mapper import map_bdns_to_brick_equipment
#from points_csv_ingestor.loader import load_points_csv
#from points_linker.linker import link_points_to_equipment
#from topology_reasoner.reasoner import build_graph
#from rdf_writer.writer import write_turtle

In [ ]:
# IFCファイルパスの入力
#異なるファイルパスを入力する場合，座標系が一致していることを確認してください。
ifc_path_MEP = input("Enter IFC file path for MEP model including BDNS Information: ")

ifc_path_SPACE = input("Enter IFC file path for model including IfcSpace: ")

In [19]:
bundle_MEP = load_ifc_bundle(str(ifc_path_MEP))
bundle_SPACE = load_ifc_bundle(str(ifc_path_SPACE))

print("\n=== IFC Loaded for MEP===")
print("schema:", bundle_MEP.schema)
print("model type:", type(bundle_MEP.model))

print("\n=== IFC Loaded for SPACE===")
print("schema:", bundle_SPACE.schema)
print("model type:", type(bundle_SPACE.model))


=== IFC Loaded for MEP===
schema: IFC2X3
model type: <class 'ifcopenshell.file.file'>

=== IFC Loaded for SPACE===
schema: IFC2X3
model type: <class 'ifcopenshell.file.file'>


In [20]:
# --- BDNS タグ抽出 ---
from bdns_extractor.extractor import extract_bdns_tags, extract_spatial_elements
bdns_assets = extract_bdns_tags(bundle_MEP)

print("\n=== BDNS Extraction Result ===")
print("BDNS-tagged assets:", len(bdns_assets.items))

# 最初の数件を表示
for asset in bdns_assets.items[:10]:
    print(
        "GUID:", asset.ifc_guid,
        "| BDNS:", asset.bdns_tag,
        "| Name:", asset.name,
        "| Class:", asset.raw_ifc_class
    )


=== BDNS Extraction Result ===
BDNS-tagged assets: 6511
GUID: 2HwPth0ED3muP_ET7DKBaz | BDNS: outlet - double switched socket outlet | Name: DSSO-1 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK99g | BDNS: outlet - double switched socket outlet | Name: DSSO-2 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9CM | BDNS: outlet - double switched socket outlet | Name: DSSO-3 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9Ws | BDNS: outlet - double switched socket outlet | Name: DSSO-4 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9_L | BDNS: outlet - double switched socket outlet | Name: DSSO-5 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7lq | BDNS: outlet - double switched socket outlet | Name: DSSO-6 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7qD | BDNS: outlet - double switched socket outlet | Name: DSSO-7 | Class: IfcFlowTerminal
GUID: 3ID7pTUZ96xeg742Mq2AA7 | BDNS: outlet - double switched socket outlet | Name: DSSO-8 | Class: IfcFlowTerminal
GUID: 2Rg2Qh64n

In [22]:
import json
# ============ 空間要素抽出と JSON 出力 ============
from bdns_extractor.extractor import extract_spatial_elements

spatial = extract_spatial_elements(bundle_SPACE)
with open("spatial_elements.json", "w", encoding="utf-8") as f:
    json.dump(
        [e.__dict__ for e in spatial.items],
        f,
        ensure_ascii=False,
        indent=2
    )

In [23]:
spatial

SpatialElements(items=[SpatialElement(ifc_guid='3Q7I9g28nFrf6cK_3TaT4h', name='51 Moorgate', raw_ifc_class='IfcSite', composition_type='ELEMENT', elevation=None, parent_guid='3Q7I9g28nFrf6cK_3TaT4f', path='IfcProject[51M]/IfcSite[51 Moorgate]'), SpatialElement(ifc_guid='1yHpNbHTP0jwQ9b6wJsu0f', name='Floor Drain:Floor Drain:1228334', raw_ifc_class='IfcSite', composition_type='PARTIAL', elevation=None, parent_guid=None, path='IfcSite[Floor Drain:Floor Drain:1228334]'), SpatialElement(ifc_guid='3Q7I9g28nFrf6cK_3TaT4e', name='51 Moorgate', raw_ifc_class='IfcBuilding', composition_type='ELEMENT', elevation=None, parent_guid='3Q7I9g28nFrf6cK_3TaT4h', path='IfcProject[51M]/IfcSite[51 Moorgate]/IfcBuilding[51 Moorgate]'), SpatialElement(ifc_guid='1qglsiRyn1nBpBBHVjIpGH', name='B1', raw_ifc_class='IfcBuildingStorey', composition_type='ELEMENT', elevation=5835.0, parent_guid='3Q7I9g28nFrf6cK_3TaT4e', path='IfcProject[51M]/IfcSite[51 Moorgate]/IfcBuilding[51 Moorgate]/IfcBuildingStorey[B1]'), Sp

In [24]:
from class_mapper.mapper import map_bdns_to_brick_equipment

base_ns = "http://example.com/brick#"

In [25]:
from pathlib import Path
from class_mapper.mapper import map_bdns_to_brick_equipment

# Define CSV path relative to project root
mapping_csv_path = PROJECT_ROOT / "data" / "resources" / "bdns_assets_to_brick_map.csv"

base_ns = "http://example.com/brick#"

equip_set = map_bdns_to_brick_equipment(
    tagged_assets=bdns_assets,
    base_ns=base_ns,
    crosswalk_equip=None,
    mapping_csv_path=str(mapping_csv_path),
)

Mapping CSV not found. path=c:\Users\Owner\Documents\25952501_RioSaijo\IFC2RDF_Fork\data\resources\bdns_assets_to_brick_map.csv
